# Lab 2: Fine-Tuning with LoRA (and QLoRA)

**Time:** about 60 minutes

In Lab 1 you built the training examples. Now use them to fine-tune a small language model, so it answers in the Fund's
way.

### Why not just train the whole model?

**Full fine-tuning** updates every weight in the model. For a 7-billion-parameter model that means holding the model,
its gradients and the optimiser's state in memory at once, which comes to several data-centre GPUs. It also produces a
complete new copy of the model for every experiment.

**LoRA** (Low-Rank Adaptation) takes a different route. It freezes the original model and trains small extra matrices
beside some of its layers. Typically well under 1% of the parameters are trained, and the result is an **adapter** of a
few megabytes that sits on top of the unchanged base model. That's what makes fine-tuning possible on a laptop, and it
means you can keep several adapters (one for member services, one for compliance) on one shared base model.

An everyday comparison: instead of rewriting a 1,000-page procedures manual for each department, you keep one manual
and give each department a thin booklet of corrections and additions. LoRA's adapter is that booklet.

```
                 frozen base model (hundreds of millions of weights, unchanged)
 input ─► ... ─► [ attention weights W ] + [ B × A ] ─► ... ─► output
                                             └── LoRA: two small trainable matrices, rank r
```

**QLoRA** goes one step further: it loads the frozen base model in 4-bit precision, so bigger models fit on smaller GPUs.
Section 8 shows how.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **base model** | the pre-trained model you start from; it stays unchanged |
| **full fine-tuning** | updating every weight in the model |
| **LoRA** | Low-Rank Adaptation: training two small matrices beside each chosen layer, with the base frozen |
| **adapter** | the small file of LoRA weights that sits on top of the base model |
| **frozen** | weights that are not changed during training |
| **rank (r)** | the size of LoRA's small matrices: bigger means more capacity to learn |
| **PEFT** | parameter-efficient fine-tuning: methods like LoRA that train only a small part of a model |
| **QLoRA** | LoRA on a base model stored in 4-bit numbers, to save memory |
| **label -100** | a marker meaning "don't learn from this token" |
| **training / validation loss** | how wrong the model is on training examples / on unseen examples; lower is better |
| **overfitting** | memorising the training examples instead of learning the general pattern |
| **probe questions** | a few fixed questions asked before and after training to see what changed |
| **bfloat16 / float16** | 16-bit number formats that halve memory compared with 32-bit |

## Where this lab fits

**What you've done so far.** In Lab 1 you built a clean instruction dataset: corrected FAQ answers and behaviour examples
in chat format, split so that paraphrases can't leak, saved to `data/train.jsonl` and `data/val.jsonl`. On Day 6 you
fine-tuned a transformer classifier with the Hugging Face `Trainer`, and on Day 7 you looked inside
Qwen2.5-0.5B-Instruct and watched it predict the next token.

**What you'll do now.** You'll fine-tune that same small model on your dataset with LoRA. You'll record its answers to
four probe questions first, turn each conversation into a training example that only learns from the answer, add LoRA
adapters and count how few weights are trained, train while watching both losses, compare the answers before and
after, and save the adapter with a record of how it was made.

**Why it matters.** Fine-tuning is how an organisation adapts a general model to its own voice, format and behaviour.
LoRA makes that affordable: a laptop instead of a data centre, and a few megabytes per adapter instead of a full copy
of the model.

**What happens next.** Lab 3 tests the adapter properly against the base model on the held-out questions, and checks
that nothing important got worse.

### Theory: where the memory goes during training

Training needs much more memory than using a model. For every trainable weight, the GPU has to hold:

| Item | Why it's needed | Typical size per weight |
|---|---|---|
| the weight itself | to compute the output | 2 bytes (16-bit) |
| its gradient | which way to move it (Day 3) | 2-4 bytes |
| the optimiser's state | Adam keeps two running averages per weight (Day 3) | 8 bytes |

So full fine-tuning needs roughly **12-16 bytes per parameter**, plus memory for the activations. For a 7-billion-
parameter model that's about 100 GB: several data-centre GPUs.

LoRA changes the sums. The base model's weights are **frozen**, so they need no gradients and no optimiser state: just
the 2 bytes each to hold them. Only the small adapter matrices (well under 1% of the parameters) carry the full training
cost. QLoRA goes further and stores the frozen weights in 4 bits (half a byte each). That's why the table in Section 8
shows a 7B model fitting on a single 12 GB GPU.

## Imports

New this time:

- `peft`: Hugging Face's library for LoRA and other parameter-efficient fine-tuning methods. `LoraConfig` holds the
  settings; `get_peft_model` adds the adapters to a model;
- `DataCollatorForSeq2Seq`: pads each batch, including the labels.

The base model is **Qwen2.5-0.5B-Instruct**, the same small model you looked inside on Day 7: small enough to fine-tune on
a laptop in minutes.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

# peft: Hugging Face's library for LoRA and other "parameter-efficient fine-tuning" methods.
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

Check the hardware. Three settings depend on it:

- `GPU`: is there an NVIDIA GPU?
- `BF16`: does it support **bfloat16**, a 16-bit number format that newer GPUs handle well and that is less likely to
  overflow during training than ordinary float16?
- `QUICK_RUN`: without a GPU, train for fewer epochs so the lab finishes in reasonable time.

In [ ]:
# What hardware do we have?
GPU = torch.cuda.is_available()

# bfloat16 is a 16-bit number format that newer GPUs handle well. Use it if the GPU supports it.
BF16 = GPU and torch.cuda.is_bf16_supported()

# On a CPU: fewer epochs so the lab finishes in time.
QUICK_RUN = not GPU

if GPU:
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16)
else:
    print("GPU: none (QUICK_RUN on)")

> **Where we are.** The libraries are loaded, and you know whether there's a GPU, whether it supports bfloat16, and
> whether `QUICK_RUN` is on. First, the data Lab 1 saved.

## 1. The dataset from Lab 1

Lab 1 saved the training and validation conversations as JSON Lines files: one conversation per line. Read each file,
split it into lines (`splitlines()`), skip any blank line, and turn each line into a Python dictionary with
`json.loads`. If Lab 1 hasn't been run, stop with a clear message rather than a confusing error later.

In [ ]:
if not Path("data/train.jsonl").exists():
    raise FileNotFoundError("Run Lab 1 first: it writes data/train.jsonl and data/val.jsonl")

# JSON Lines: one conversation per line. Read each line with json.loads.
train_rows = []
for line in Path("data/train.jsonl").read_text(encoding="utf-8").splitlines():
    if line.strip():
        train_rows.append(json.loads(line))

val_rows = []
for line in Path("data/val.jsonl").read_text(encoding="utf-8").splitlines():
    if line.strip():
        val_rows.append(json.loads(line))

print(len(train_rows), "training and", len(val_rows), "validation conversations")

Every conversation starts with the same system prompt (Lab 1, Section 7). Keep a copy in `SYSTEM`: you'll use exactly
the same text when asking questions, because training and use must match. Then look at one conversation.

In [ ]:
# Every conversation starts with the same system prompt. Keep a copy: you'll use it when asking questions.
SYSTEM = train_rows[0]["messages"][0]["content"]

print(json.dumps(train_rows[5], indent=1, ensure_ascii=False)[:600])      # the first 600 characters

> **Where we are.** The training and validation conversations are loaded, and `SYSTEM` holds the system prompt they all
> share. Before changing the model, record how it behaves now.

## 2. Load the base model, and ask it some questions first

Always record how the model behaves **before** training, on a few questions you'll ask again afterwards. Without a
"before", you can't tell what the training did.

Load the tokenizer and the model as on Day 7. The number format depends on the hardware: bfloat16 if the GPU supports
it, float16 on other GPUs, and ordinary 32-bit float on a CPU (CPUs are slow with 16-bit numbers).

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

# The number format for the weights: bfloat16 or float16 on a GPU, float32 on a CPU.
if BF16:
    dtype = torch.bfloat16
elif GPU:
    dtype = torch.float16
else:
    dtype = torch.float32

device = "cuda" if GPU else "cpu"
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype).to(device)

# Count the parameters.
n_parameters = 0
for parameter in model.parameters():
    n_parameters = n_parameters + parameter.numel()
print(BASE_MODEL, ":", round(n_parameters / 1e6), "million parameters")

`ask` sends one question to a model, with the training system prompt, and returns the answer. The steps are the ones you
did by hand on Day 7: wrap the messages in the chat template, turn them into token ids, `generate` (with
`do_sample=False`, like temperature 0), and decode only the new tokens. `pad_token_id=tok.eos_token_id` avoids a warning
about padding.

In [ ]:
def ask(model, question, max_new_tokens=120):
    """Ask the model one question with the training system prompt, and return its answer (as in Day 7)."""
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    n_prompt_tokens = inputs["input_ids"].shape[1]

    model.eval()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)

    new_tokens = output[0, n_prompt_tokens:]          # only the answer, not the prompt
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

Four **probe questions**, chosen to test different things: a fact (the funeral grant), a rule (resignation and
unemployment benefit), a behaviour (refusing a question about personal records) and Swahili. Ask each one now and keep
the answers in `before`.

In [ ]:
# Four "probe" questions you'll ask again after training.
PROBES = [
    "How much is the funeral grant?",
    "Do I get unemployment benefit if I resign?",
    "What is my contribution balance?",
    "Msaada wa mazishi ni kiasi gani?",
]

before = {}
for question in PROBES:
    before[question] = ask(model, question)
    print("Q:", question)
    print("A:", before[question])
    print()

The base model is fluent and helpful-sounding, but it doesn't know the Fund's rules, and it may happily *invent* a way to
check a balance it can't see.

> **Where we are.** The base model's answers to the four probe questions are saved in `before`: fluent, but not the Fund's
> rules. Next, prepare the conversations so the model learns the right thing from them.

### Theory: what the model is actually trained to do

An LLM is trained the same way it generates: **predict the next token**. For a training conversation, the model sees
the tokens one at a time and, at every position, gives a probability to every possible next token. The loss (Day 3's
cross-entropy) is high when it gives a low probability to the token that actually comes next. Training nudges the
weights to make the real next tokens more likely.

So "teaching the model to answer like the Fund" means: across the training answers, make each next token of the Fund's
answer more likely, given everything before it. Labels decide **which positions count**. With `-100` on the system
prompt and question, the loss only looks at the answer, so the model is rewarded for answering well, not for guessing
what members will ask.

## 3. Turn conversations into training examples

The model learns by predicting each next token of the conversation and being corrected (as on Day 7, generation is
next-token prediction). Two details matter here, and they're the ones people most often get wrong:

1. **Use the model's own chat template**, exactly as it will be used after deployment.
2. **Only train on the answer.** The system prompt and the question are there for context, but the model shouldn't be
   rewarded for predicting the question. Tokens you don't want the loss computed on get the label `-100`, which PyTorch
   ignores.

Do it for one conversation first. Build two texts:

- the **prompt part**: everything except the answer (`conversation[:-1]` = all messages but the last), ending with the
  marker where the assistant's answer starts;
- the **full text**: the whole conversation, answer included.

Turn both into token ids. The full text starts with exactly the prompt's tokens, followed by the answer's tokens.

In [ ]:
conversation = train_rows[0]["messages"]

# The prompt part (system + question), ending where the assistant's answer should start...
prompt_text = tok.apply_chat_template(conversation[:-1], tokenize=False, add_generation_prompt=True)

# ...and the whole conversation, answer included.
full_text = tok.apply_chat_template(conversation, tokenize=False)

prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
full_ids = tok(full_text, add_special_tokens=False)["input_ids"]
print("prompt tokens:", len(prompt_ids), "| whole conversation:", len(full_ids))

Now the **labels**: what the model should predict at each position. For the prompt's positions, use -100 ("ignore");
for the answer's positions, use the real token ids. `[-100] * n` makes a list of n copies of -100, and
`full_ids[len(prompt_ids):]` is "everything after the prompt", which is the answer.

Decoding just the answer's tokens checks that the split is in the right place: you should see only the answer text.

In [ ]:
# Labels: -100 for every prompt token ("don't learn from this"), then the real token ids for the answer.
labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

# Decode only the tokens the model will be trained on: it should be just the answer.
answer_ids = full_ids[len(prompt_ids):]
print("trained-on part decodes to:", repr(tok.decode(answer_ids)))

The same steps as a function, `encode`, applied to every conversation. It also cuts very long conversations at `MAX_LEN`
tokens (and cuts the labels to match), and adds an **attention mask** of 1s (every token is real; padding is added
later, per batch). A plain Python list of these dictionaries works as a dataset for the Trainer.

In [ ]:
MAX_LEN = 384          # longer conversations are cut off at 384 tokens


def encode(conversation):
    """One conversation -> token ids, attention mask, and labels with the prompt part masked out (-100)."""
    messages = conversation["messages"]
    prompt_text = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tok.apply_chat_template(messages, tokenize=False)

    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tok(full_text, add_special_tokens=False)["input_ids"][:MAX_LEN]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    labels = labels[:len(full_ids)]                      # same length as the token ids
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


# A plain Python list of encoded examples works as a dataset for the Trainer.
train_ds = []
for row in train_rows:
    train_ds.append(encode(row))

val_ds = []
for row in val_rows:
    val_ds.append(encode(row))

print(len(train_ds), "training examples,", len(val_ds), "validation examples")

> **Where we are.** Every conversation is now a dictionary of token ids, labels (with -100 on everything but the answer)
> and an attention mask. The model is still the unchanged base model. Now add the small trainable parts.

### Theory: why a small, low-rank change is enough

Fine-tuning a pre-trained model doesn't need to change everything it knows. It needs to shift its behaviour in a few
directions: shorter answers, the Fund's tone, refusals for personal records. Research found that the changes full
fine-tuning makes to each weight matrix are usually **low-rank**: they can be described by far fewer numbers than the
matrix holds.

**Rank** in plain terms: a big table of numbers can sometimes be rebuilt by multiplying two thin tables. A 896 × 896
table has about 800,000 numbers; a 896 × 16 table times a 16 × 896 table also gives a 896 × 896 table, from about
29,000 numbers. That product can't express *every* possible change, only changes of "rank 16" or less. LoRA's bet is
that the change fine-tuning needs fits in that space, and in practice it usually does.

LoRA's matrix B starts at zero, so at the start of training the adapter adds nothing and the model behaves exactly like
the base. Training then grows the change from there.

## 4. Add LoRA adapters

The settings you'll meet everywhere:

- `r` (rank): the size of the small matrices. 8-16 is usual. Higher means more capacity to learn, and more to overfit.
- `lora_alpha`: a scaling factor, commonly twice `r`. It controls how strongly the adapter's changes are applied.
- `lora_dropout`: randomly switches off 5% of the adapter's values during training, a guard against overfitting (Day 3).
- `target_modules`: which layers get adapters. Here, all the attention projections (`q_proj`, `k_proj`, `v_proj`,
  `o_proj`: the parts of attention that decide which tokens to look at, and what to take from them) and the
  feed-forward projections (`gate_proj`, `up_proj`, `down_proj`: the part of each block that processes each token).

### LoRA in numbers

One attention matrix in this model has 896 × 896 = about 800,000 weights. Full fine-tuning would update all of them.
LoRA freezes the matrix and adds two thin matrices beside it: one 896 × 16 and one 16 × 896, about 29,000 weights in
total. During training only those are updated, and their product (896 × 896 again) is added to the frozen matrix's
output. The `r=16` below is that middle size: the "rank". Smaller means fewer weights to train, larger means more
capacity to change the model's behaviour.

`get_peft_model` does the work: it freezes every original weight and adds the small matrices beside each target layer.

In [ ]:
lora_config = LoraConfig(
    r=16,                       # the rank
    lora_alpha=32,              # the scaling factor
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",      # a model that predicts the next token
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Wrap the model: the original weights are frozen, and LoRA matrices are added beside the target layers.
model = get_peft_model(model, lora_config)

Check how much is actually being trained. Every weight tensor has a `requires_grad` flag: True means "train this", False
means "frozen". Count both.

In [ ]:
# Count the trainable weights (requires_grad=True) and all the weights.
trainable = 0
total = 0
for parameter in model.parameters():
    total = total + parameter.numel()
    if parameter.requires_grad:
        trainable = trainable + parameter.numel()

print(f"trainable parameters: {trainable / 1e6:.1f} million of {total / 1e6:.0f} million ({100 * trainable / total:.2f}%)")

> **Where we are.** LoRA adapters sit beside every attention and feed-forward projection, and only a fraction of a
> percent of the weights are trainable. Now train them.

## 5. Train

With about 100 short examples, a few passes over the data (epochs) are enough. The learning rate is much higher than on
Day 6 (2e-4 vs 5e-5) because only the small adapter matrices are learning, and they start from zero.

The settings, most of them familiar from Day 6:

| Setting | Meaning |
|---|---|
| `num_train_epochs` | 5 passes over the data (2 on a CPU) |
| `per_device_train_batch_size=8` | 8 conversations per training step |
| `learning_rate=2e-4` | 0.0002 |
| `lr_scheduler_type="cosine"` | the learning rate falls smoothly towards zero as training ends |
| `warmup_steps=5` | start with smaller steps for the first 5 steps |
| `logging_steps=5` | record the training loss every 5 steps |
| `eval_strategy="epoch"` | measure the validation loss after each epoch |
| `bf16` / `fp16` | use 16-bit numbers on a GPU, to save memory and time |

The data collator pads each batch to its longest example, and pads the labels with -100 so the padding isn't learned
from either.

In [ ]:
if QUICK_RUN:
    EPOCHS = 2
else:
    EPOCHS = 5

args = TrainingArguments(
    output_dir="lora_runs",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",        # the learning rate falls smoothly towards zero
    warmup_steps=5,
    weight_decay=0.0,
    logging_steps=5,                   # record the training loss every 5 steps
    eval_strategy="epoch",             # check the validation loss after each epoch
    save_strategy="no",
    bf16=BF16,
    fp16=GPU and not BF16,
    report_to="none",
    seed=SEED,
)

# DataCollatorForSeq2Seq pads each batch to its longest example, and pads the labels with -100.
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)

Train, and time it:

In [ ]:
start = time.perf_counter()
trainer.train()
print(f"training took {(time.perf_counter() - start) / 60:.1f} minutes")

The Trainer keeps a log of everything it measured, in `trainer.state.log_history`: some rows hold the training loss
(`loss`), others the validation loss (`eval_loss`). `notna()` keeps the rows that have each one. Plot both against the
epoch.

In [ ]:
# The Trainer kept a log: training-loss rows and validation-loss rows.
logs = pd.DataFrame(trainer.state.log_history)
train_log = logs[logs["loss"].notna()]
eval_log = logs[logs["eval_loss"].notna()]

plt.figure(figsize=(8, 3.5))
plt.plot(train_log["epoch"], train_log["loss"], color="#2a78d6", label="training loss")
plt.plot(eval_log["epoch"], eval_log["eval_loss"], color="#eb6834", marker="o", label="validation loss")
plt.title("LoRA fine-tuning")
plt.xlabel("epoch")
plt.legend()
plt.show()

Training loss should fall steadily. Watch the validation loss: if it starts to rise while training loss keeps falling, the
adapter is memorising the training answers (Day 3's overfitting, again). With a dataset this small, that can happen within
a few epochs.

> **Where we are.** The adapter has trained, and the loss curves show how training and validation loss moved epoch by
> epoch. A falling loss doesn't tell you whether the answers are better. The probes do.

## 6. Before and after

Ask the four probe questions again, now with the adapter in place, and put the answers side by side with the "before"
answers. `[before[q] for q in PROBES]` is a short way of writing "the before-answer for each probe, in order".

In [ ]:
after = {}
for question in PROBES:
    after[question] = ask(model, question)

comparison = pd.DataFrame({
    "question": PROBES,
    "before": [before[q] for q in PROBES],
    "after": [after[q] for q in PROBES],
})
comparison

Go through the table row by row, and be as hard on the "after" column as on the "before" column. On a model this small,
trained on about a hundred examples, you'll typically see a mixture:

- **style** changes almost at once: shorter, plainer answers in the Fund's voice;
- some **facts that were in the training data** come back correctly (the funeral grant amount is a good one to check);
- some answers get **worse**. On one test run, the base model answered the resignation question correctly ("No"), and after
  fine-tuning it answered "Yes", in the same confident tone. Fine-tuning nudges every answer towards the patterns in the
  training data, and with little data those nudges can land in the wrong place;
- **behaviour** (refusing the balance question) and **Swahili** only improve if there were enough examples of them. A
  handful often isn't enough for a 0.5B model.

None of this is visible from the loss curve, which looked fine. It only shows up when you compare answers, which is exactly
why Lab 3 exists.

**Your turn (1).** Ask the fine-tuned model three questions that are **not** in the FAQ but *are* answered in the
handbook, for example about survivors' pension shares or invalidity reviews. Does it answer correctly, or does it answer
in the Fund's confident style... and get the facts wrong? Keep that in mind for Lab 3.

In [ ]:
# Your code here

> **Where we are.** You've compared the before and after answers honestly: style changed quickly, some trained facts came
> back, some answers got worse, and behaviour and Swahili depend on how many examples there were. Save the adapter so Lab 3
> can test it properly.

## 7. Save the adapter

`save_pretrained` on a LoRA-wrapped model saves **only the adapter**: the small matrices, plus a note of which base model
they belong to. Save the tokenizer alongside it. The weights are stored in `.safetensors` files; adding up their sizes
shows how small the adapter is.

In [ ]:
ADAPTER_DIR = Path("adapters/member_service_lora")

# Saves only the small LoRA matrices (and the tokenizer), not the whole base model.
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)

size_mb = 0
for f in ADAPTER_DIR.glob("*.safetensors"):
    size_mb = size_mb + f.stat().st_size / 1e6      # bytes -> megabytes
print(f"adapter saved to {ADAPTER_DIR}: {size_mb:.1f} MB (the base model is about 990 MB and is not copied)")

Save a small record of how the adapter was made next to it: the base model, the LoRA settings, the training settings, the
data it was trained on, and the final validation loss. Like Lab 1's data card, it answers the questions someone will ask
months from now ("which data was this trained on?").

In [ ]:
# A small record of how the adapter was made.
training_info = {
    "base_model": BASE_MODEL,
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05},
    "epochs": EPOCHS,
    "learning_rate": 2e-4,
    "train_examples": len(train_ds),
    "val_examples": len(val_ds),
    "data_card": "data/data_card.md",
    "final_eval_loss": float(eval_log["eval_loss"].iloc[-1]),        # the last validation loss
}
Path(ADAPTER_DIR, "training_info.json").write_text(json.dumps(training_info, indent=2))
training_info

The adapter records which base model it belongs to. To use it, you load the base model and then the adapter:

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = PeftModel.from_pretrained(base, "adapters/member_service_lora")
```

> **Where we are.** The adapter is saved (a few megabytes), with its tokenizer and a record of how it was made. Everything
> so far used a 0.5B model. Real use would need a bigger one.

## 8. Going bigger: QLoRA

A 0.5B model is ideal for learning the mechanics. For real use you'd fine-tune something larger, from the shortlist in
Day 7's scorecard. QLoRA makes that feasible on a single GPU (graphics processing unit) by loading the frozen base model
in 4-bit precision (the same idea as the quantised models in Ollama on Day 7):

| Base model | LoRA, 16-bit base | QLoRA, 4-bit base |
|---|---|---|
| 0.5-1.5B | 4-8 GB GPU | 3-4 GB GPU |
| 3B | about 12 GB | about 6 GB |
| 7-8B | about 24 GB | about 10-12 GB |

(Rough figures for short sequences and small batches.) The code changes are small:

- `BitsAndBytesConfig` describes the 4-bit loading: `nf4` is a 4-bit format designed for model weights, and "double
  quantisation" squeezes out a little more memory;
- `prepare_model_for_kbit_training` gets the 4-bit model ready for training;
- `device_map="auto"` lets the library place the model on the GPU.

Set `USE_QLORA = True` to run it if you have an NVIDIA GPU with 8 GB or more and `bitsandbytes` installed:

In [ ]:
USE_QLORA = False
QLORA_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

if USE_QLORA and GPU:
    from peft import prepare_model_for_kbit_training
    from transformers import BitsAndBytesConfig

    # Load the base model with its weights stored in 4 bits ("nf4" is a 4-bit format designed for model weights).
    quantisation = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
    )
    q_model = AutoModelForCausalLM.from_pretrained(QLORA_MODEL, quantization_config=quantisation, device_map="auto")
    q_model = prepare_model_for_kbit_training(q_model)

    # The same LoRA settings as before.
    q_model = get_peft_model(q_model, lora_config)
    q_model.print_trainable_parameters()
    # ...then the same Trainer as above, using a tokenizer and datasets built with QLORA_MODEL's chat template.
else:
    print("QLoRA skipped. Set USE_QLORA = True on a machine with an NVIDIA GPU and bitsandbytes installed.")

> **Where we are.** You've seen how QLoRA fits bigger models onto a single GPU by storing the frozen base in 4 bits. The
> last question is how a fine-tuned model would reach colleagues.

## 9. From adapter to something colleagues can use

To serve the fine-tuned model through Ollama, like the models on Day 7:

1. **Merge** the adapter into the base model: `merged = model.merge_and_unload(); merged.save_pretrained("merged_model")`.
   This adds the adapter's changes into the base weights permanently, giving one ordinary model again.
2. **Convert** to Ollama's GGUF format with the `convert_hf_to_gguf.py` script from the llama.cpp project, and quantise
   it to 4-bit.
3. **Create** an Ollama model from it with a `Modelfile` that sets the same system prompt you trained with.

Before any of that, Lab 3 has to show the fine-tuned model is actually better, and that nothing important got worse.

---

## Looking back, looking ahead

**What you did in this lab.** You fine-tuned Qwen2.5-0.5B-Instruct with LoRA: recorded the base model's answers to probe
questions, built training examples that only learn from the answer, added adapters to the attention and feed-forward
layers, trained a tiny fraction of the weights while watching training and validation loss, compared answers before and
after, saved the adapter with a record of its making, and saw how QLoRA and merging take this to bigger models and to
Ollama.

**The ideas to carry forward.**

- Frozen weights need no gradients or optimiser state; that's where LoRA's memory saving comes from.
- Train on the answer tokens only (labels of -100 elsewhere).
- Always record "before" so you can see what training changed.
- A good-looking loss curve can hide worse answers.
- When results disappoint, look at the data first.

**What happens next.** Lab 3 answers the real question: on staff-written questions the model never saw, is the
fine-tuned model better than the base model, and did anything get worse?

## Notes on the settings

- If the model barely changes, train for more epochs or raise the learning rate a little.
- If it starts repeating training answers word for word to unrelated questions, it's overfitting: fewer epochs, lower `r`,
  or more varied data.
- Most of the quality comes from the **data** in Lab 1, not from these settings. When results disappoint, look at the data
  first.